# ⚔️ ClashBot: AlphaStar + OpenAI Five Self-Play Training on Google Colab
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/varun-zero/clash-ml-model/blob/main/ClashBot_Colab_Training.ipynb)


Welcome to the **ClashBot Pure Neural Self-Play Training Pipeline**.

### 🧠 Neural RL Architecture
- **Core Model**: `AlphaStarCRNet` (Spatial Residual ResNet + Transformer Entity Encoder + PPO Autoregressive Heads)
- **100% Pure Self-Play**: 0% scripted bots. The model trains strictly against prioritized historical snapshots (70%) and live mirror self-play (30%).
- **Authentic Match Tracking**: Real-time wins, losses, draws, and win rate derived directly from `crsim` engine simulation (zero synthetic numbers).
- **Google Drive Persistence**: Auto-saves `latest.pt`, `training_history.json`, and `clashbot_training_curve.png` to Google Drive.

## 1. Verify Tesla T4 GPU & CUDA Acceleration

In [ ]:
# Verify GPU allocation (Tesla T4 16GB VRAM recommended)
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 2. Mount Google Drive for Continuous Persistence

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/clashbot_training'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"✅ Google Drive Mounted! Checkpoint sync path: {DRIVE_DIR}")

## 3. Clone Repository & Install Dependencies

In [ ]:
# Clone repository and install dependencies
!git clone https://github.com/varun-zero/clash-ml-model.git /content/clashbot
%cd /content/clashbot
!pip install -q -r requirements.txt

import os, sys
cwd = os.getcwd()
if cwd not in sys.path:
    sys.path.insert(0, cwd)
print(f"Working Directory: {cwd}")

## 4. Run Pure Neural Self-Play Training (0% Scripted Bots)

Runs PPO training with:
- 8 parallel environments
- 256 rollout steps per environment (2048 samples per batch)
- FP16 AMP mixed precision
- 100% Neural Self-Play against League snapshots and mirror policy
- Live match record (Wins / Losses / Draws) tracking
- Auto-sync to Google Drive every 5 updates

In [ ]:
!python train_colab.py \
    --total-updates 1000 \
    --num-envs 8 \
    --rollout-steps 256 \
    --checkpoint-dir ./checkpoints \
    --drive-dir /content/drive/MyDrive/clashbot_training

## 5. View Real-Time Improvement & Telemetry Curves

In [ ]:
from IPython.display import Image, display
import os

plot_path = "checkpoints/clashbot_training_curve.png"
if os.path.exists(plot_path):
    display(Image(plot_path))
else:
    drive_plot = "/content/drive/MyDrive/clashbot_training/clashbot_training_curve.png"
    if os.path.exists(drive_plot):
        display(Image(drive_plot))
    else:
        print("Training plot will be generated after the first PPO batch!")